# Protocoles de dialogue de Walton-Krabbe — types, actes, transitions, terminaison

**Notebook pédagogique CoursIA** · #1961 Phase 5 (assets CoursIA réutilisables) · `Claude Code @ myia-po-2025:2025-Epita-Intelligence-Symbolique`.

Corpus-free : tous les exemples sont synthétiques, domaine-public. Aucun LLM, aucune JVM — un protocole de dialogue est une machine à états déterministe.

Ce que couvre ce notebook :

1. les **6 types de dialogue** de la taxonomie Walton-Krabbe et les **9 actes de langage** formalisés ;
2. les **règles de transition** — ce qu'un protocole permet comme réponse à chaque acte, et l'**asymétrie clé** `CLAIM→CONCEDE` (interdit en inquiry, permis en persuasion) ;
3. les **conditions de terminaison** — concession finale, double retrait, boucle de motif ;
4. la **jonction avec l'asset 1** : peupler `FormalArgument.scheme` avec le classifieur de schémas de Walton.

**Garde round-trip** : chaque transition et chaque verdict de terminaison affiché ici est rejoué contre le moteur source par `tests/unit/coursia/dialogue_protocols/test_dialogue_protocols_roundtrip.py` (mêmes exemples, même JSON partagé) — ce notebook ne peut pas dériver silencieusement.


## §1 — Le modèle : 6 types de dialogue, 9 actes, une machine à états

Un **type de dialogue** (inquiry, persuasion, negotiation, deliberation, eristic, information seeking) fixe l'objectif de l'échange ; les **actes de langage** (claim, question, challenge, argue, concede, retract, support, refute, understand) sont les coups jouables. Un protocole concret = une table de transitions autorisées + des conditions de terminaison.


In [1]:
# Imports + configuration. Les protocoles sont déterministes (aucun LLM, aucune JVM).
import json
import logging
import os
import sys

logging.disable(logging.INFO)  # sorties propres malgré la chaîne d'import du dépôt

# Localiser la racine du dépôt (contient argumentation_analysis/) en remontant depuis le CWD.
_cwd = os.getcwd()
while _cwd and not os.path.isdir(os.path.join(_cwd, "argumentation_analysis")):
    _parent = os.path.dirname(_cwd)
    if _parent == _cwd:
        break
    _cwd = _parent
ROOT = _cwd if os.path.isdir(os.path.join(_cwd, "argumentation_analysis")) else os.getcwd()
sys.path.insert(0, ROOT)

from argumentation_analysis.agents.core.debate.protocols import (
    DialogueMove,
    DialogueType,
    InquiryProtocol,
    PersuasionProtocol,
    SpeechAct,
)

EXAMPLES_PATH = os.path.join(
    ROOT, "docs", "coursia_contrib", "dialogue_protocols_examples.json"
)

print("Types de dialogue (Walton-Krabbe) :")
for t in DialogueType:
    print(f"  {t.value:<20} {t.name}")

print("\nActes de langage :")
for a in SpeechAct:
    print(f"  {a.value:<12} {a.name}")

inquiry = InquiryProtocol()
persuasion = PersuasionProtocol()
print(f"\nInquiry    : {len(inquiry.allowed_transitions)} actes sources, "
      f"{len(inquiry.termination_conditions)} conditions de terminaison")
print(f"Persuasion : {len(persuasion.allowed_transitions)} actes sources, "
      f"{len(persuasion.termination_conditions)} conditions de terminaison")


Types de dialogue (Walton-Krabbe) :
  information_seeking  INFORMATION_SEEKING
  inquiry              INQUIRY
  persuasion           PERSUASION
  negotiation          NEGOTIATION
  deliberation         DELIBERATION
  eristic              ERISTIC

Actes de langage :
  claim        CLAIM
  question     QUESTION
  challenge    CHALLENGE
  argue        ARGUE
  concede      CONCEDE
  retract      RETRACT
  support      SUPPORT
  refute       REFUTE
  understand   UNDERSTAND

Inquiry    : 8 actes sources, 4 conditions de terminaison
Persuasion : 9 actes sources, 3 conditions de terminaison


Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
2026-09-14 17:48:22 [WARNING] [Services.CryptoService] crypto_service.__init__:46 - Service de chiffrement initialisé sans clé. Le chiffrement est désactivé.


## §2 — Transitions : ce que chaque protocole permet (et interdit)

La table `is_valid_move(depuis, vers)` est la règle du jeu. Le point le plus instructif est l'**asymétrie** `CLAIM→CONCEDE` : en inquiry, on ne **concède** pas une affirmation (on la soutient, la conteste ou la questionne) ; en persuasion, la concession est au contraire la **sortie normale** — c'est l'objectif du dialogue.


In [2]:
# Les 9 transitions du JSON partagé (avec la garde round-trip) passent au moteur réel.
with open(EXAMPLES_PATH, encoding="utf-8") as fh:
    examples = json.load(fh)

protos = {"inquiry": inquiry, "persuasion": persuasion}
for ex in examples["transitions"]:
    proto = protos[ex["dialogue"]]
    got = proto.is_valid_move(SpeechAct[ex["from"]], SpeechAct[ex["to"]])
    assert got == ex["allowed"], (ex, got)
    verdict = "autorisé" if got else "interdit"
    print(f"{ex['dialogue']:<11} {ex['from']} -> {ex['to']:<11} {verdict}")
    print(f"   {ex['why']}\n")

print("9/9 — chaque transition rend le verdict attendu.")


inquiry     QUESTION -> CLAIM       autorisé
   En inquiry, une question peut recevoir une affirmation directe.

inquiry     CLAIM -> CONCEDE     interdit
   Asymétrie clé : en inquiry on ne concède pas une affirmation — on la soutient, la conteste ou la questionne.

inquiry     CHALLENGE -> ARGUE       autorisé
   Une contestation appelle un argument.

inquiry     UNDERSTAND -> UNDERSTAND  autorisé
   La compréhension mutuelle peut s'enchaîner.

persuasion  CLAIM -> CONCEDE     autorisé
   En persuasion au contraire, concéder une affirmation est la sortie normale — l'objectif du dialogue.

persuasion  CHALLENGE -> RETRACT     autorisé
   Contesté, le locuteur peut retirer sa position.

persuasion  ARGUE -> REFUTE      autorisé
   Un argument peut être réfuté de front.

persuasion  QUESTION -> CHALLENGE   interdit
   Une question n'appelle pas une contestation directe — d'abord une réponse (claim/argue/support).

persuasion  CONCEDE -> ARGUE       interdit
   Après une concession, on r

## §3 — Terminaison : quand le dialogue s'arrête

Chaque protocole porte des **conditions de terminaison** évaluées sur l'historique : convergence (compréhensions/consécutions consécutives en inquiry), concession finale (persuasion), double retrait, boucle de motif répétée trois fois — le dialogue qui tourne en rond est coupé. Un historique sain rend `False` : le verdict est une mesure, pas une impression.


In [3]:
# Les 7 historiques du JSON partagé passent au moteur réel.
def historique(acts):
    return [DialogueMove(speaker=f"loc{i % 2}", act=SpeechAct[a], content="—")
            for i, a in enumerate(acts)]

for ex in examples["terminations"]:
    proto = protos[ex["dialogue"]]
    got = proto.is_terminal_state(historique(ex["acts"]))
    assert got == ex["terminal"], (ex, got)
    verdict = "TERMINAL" if got else "en cours"
    print(f"{ex['dialogue']:<11} {verdict:<9} {' -> '.join(ex['acts'])}")
    print(f"   {ex['why']}\n")

print("7/7 — chaque historique rend le verdict attendu.")


inquiry     TERMINAL  UNDERSTAND -> UNDERSTAND -> UNDERSTAND
   Trois compréhensions/consécutions consécutives en fin d'historique : dialogue clos.

inquiry     TERMINAL  CLAIM -> QUESTION -> UNDERSTAND -> UNDERSTAND
   Deux compréhensions consécutives en position 4 : condition de convergence atteinte.

inquiry     TERMINAL  CLAIM -> SUPPORT -> CLAIM -> SUPPORT -> CLAIM -> SUPPORT
   Boucle de motif (claim, support) répétée trois fois : le dialogue tourne en rond, on coupe.

inquiry     en cours  QUESTION -> CLAIM -> SUPPORT -> CHALLENGE -> ARGUE
   Historique sain et mixte : aucune condition de terminaison ne s'applique.

persuasion  TERMINAL  CLAIM -> CHALLENGE -> ARGUE -> CONCEDE
   Une concession en dernière position termine la persuasion — l'objectif est atteint.

persuasion  TERMINAL  CLAIM -> CHALLENGE -> RETRACT -> RETRACT
   Deux retraits consécutifs : plus rien à persuader.

persuasion  en cours  CLAIM -> CHALLENGE -> ARGUE
   Persuasion en cours, pas de concession ni retrait

## §4 — Jonction avec l'asset 1 : peupler `FormalArgument.scheme`

Le protocole laisse `FormalArgument.scheme` vide — un argument structuré sait *quoi* il conclut, pas *sous quel schéma* il avance. Le classifieur déterministe de l'asset 1 (`classify_scheme`) ferme exactement cette brèche : même stdlib, même déterminisme.


In [4]:
# Un FormalArgument dont le schéma est peuplé par le classifieur Walton (asset 1).
from argumentation_analysis.agents.core.debate.argumentation_schemes import classify_scheme
from argumentation_analysis.agents.core.debate.protocols import FormalArgument, Proposition

TEXTE = "Selon un chercheur spécialiste du domaine, la méthode est fiable."
scheme = classify_scheme(TEXTE)
assert scheme is not None and scheme.key == "expert_opinion", scheme

argument = FormalArgument(
    premises=[Proposition(content="Le chercheur est spécialiste du domaine", confidence=0.9)],
    conclusion=Proposition(content="La méthode est fiable", confidence=0.8),
    scheme=scheme.key,
)
print(f"Argument   : {argument}")
print(f"Schéma     : {scheme.key} (force a priori {scheme.strength:.2f})")
print(f"Identifiant: {argument.id[:8]}…")


Argument   : [Le chercheur est spécialiste du domaine] -> La méthode est fiable
Schéma     : expert_opinion (force a priori 0.80)
Identifiant: 7a9a17be…


## Ce qu'il faut retenir

- **6 types de dialogue** × **9 actes de langage** : le vocabulaire formel d'un échange argumentatif ;
- un protocole = une **table de transitions** + des **conditions de terminaison** — une machine à états déterministe, testable ;
- l'**asymétrie** `CLAIM→CONCEDE` : la même paire d'actes change de légalité selon l'objectif du dialogue (inquiry vs persuasion) — le type n'est pas une étiquette, il change les coups jouables ;
- la terminaison est **mesurée sur l'historique** (convergence, concession, retraits, boucle de motif) ;
- `FormalArgument.scheme` se peuple avec le classifieur de Walton (asset 1) : protocole et schéma se rejoignent.

Ce notebook est exécuté — les sorties ci-dessus sont réelles, produites par le moteur du dépôt, pas retouchées.

**Références** : Walton & Krabbe, *Commitment in Dialogue* · moteur : `argumentation_analysis/agents/core/debate/protocols.py` (adaptation du livrable étudiant `1_2_7_argumentation_dialogique`) · asset 1 : `argumentation_schemes.ipynb` · issue #1961 Phase 5 · garde : `tests/unit/coursia/dialogue_protocols/test_dialogue_protocols_roundtrip.py`.
